# CS1-EXP1-RF: SVD + Static Features + Random Forest

## Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## Define project and output paths

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData"
)

RAW_DIR = DRIVE_ROOT / "raw"
PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_DIR = DRIVE_ROOT / "manifests"

OUTPUT_DIR = DRIVE_ROOT / "outputs" / "cs1_exp1_rf"
STATIC_FEATURE_DIR = PROCESSED_DIR / "static_features"

for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    MANIFEST_DIR,
    OUTPUT_DIR,
    STATIC_FEATURE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Drive data root:", DRIVE_ROOT)
print("Processed data directory:", PROCESSED_DIR)
print("Manifest directory:", MANIFEST_DIR)
print("Static-feature cache:", STATIC_FEATURE_DIR)
print("EXP-1 output directory:", OUTPUT_DIR)

Drive data root: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData
Processed data directory: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed
Manifest directory: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests
Static-feature cache: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/static_features
EXP-1 output directory: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_exp1_rf


## Clone or refresh the GitHub repository

In [ ]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
BRANCH = "prashant"
REPO_DIR = Path("/content/DiverseVul--IS-Project")

if not REPO_DIR.exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    os.chdir(REPO_DIR)
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}

PROJECT_DIR = REPO_DIR / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.chdir(PROJECT_DIR)

print("Repository:", REPO_DIR)
print("Project directory:", PROJECT_DIR)
print("Current working directory:", Path.cwd())

Cloning into '/content/DiverseVul--IS-Project'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 54 (delta 5), reused 44 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 599.51 KiB | 5.88 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Repository: /content/DiverseVul--IS-Project
Project directory: /content/DiverseVul--IS-Project/vuln-detection
Current working directory: /content/DiverseVul--IS-Project/vuln-detection


## Install required Python packages

In [ ]:
!pip -q install \
    numpy \
    pandas \
    scipy \
    scikit-learn \
    matplotlib \
    pyyaml \
    pyarrow \
    joblib

## Verify the static-feature module

In [ ]:
import case_study_1.exp1.static_features as static_features

print("Static feature version:", static_features.STATIC_FEATURE_VERSION)
print("Number of static features:", len(static_features.FEATURE_COLUMNS))
print("\nFeature columns:")
print(static_features.FEATURE_COLUMNS)

Static feature version: cs1-static-features-v1
Number of static features: 54

Feature columns:
['raw_char_count', 'raw_line_count', 'nonempty_line_count', 'avg_nonempty_line_length', 'max_line_length', 'comment_char_count', 'comment_line_count', 'comment_char_ratio', 'string_literal_count', 'char_literal_count', 'preprocessor_directive_count', 'identifier_count', 'unique_identifier_count', 'identifier_diversity', 'numeric_literal_count', 'function_call_count', 'unique_function_call_count', 'if_count', 'else_count', 'for_count', 'while_count', 'do_count', 'switch_count', 'case_count', 'goto_count', 'return_count', 'break_continue_count', 'boolean_operator_count', 'ternary_operator_count', 'cyclomatic_complexity_proxy', 'brace_open_count', 'brace_close_count', 'parenthesis_open_count', 'array_access_count', 'star_operator_count', 'arrow_operator_count', 'address_of_operator_count', 'cast_like_count', 'pointer_declaration_count', 'integer_type_token_count', 'assignment_operator_count', 'c

## Load the frozen dataset and split manifests

In [ ]:
import pandas as pd
from pathlib import Path

NORMALIZED_DATA_PATH = (
    PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"
)

MANIFEST_PATH = (
    MANIFEST_DIR / "cs1_project_grouped_5fold_manifest.parquet"
)

HOLDOUT_MANIFEST_PATH = (
    MANIFEST_DIR / "cs1_holdout_manifest.parquet"
)

for required_path in [NORMALIZED_DATA_PATH, MANIFEST_PATH, HOLDOUT_MANIFEST_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required frozen artifact not found: {required_path}\n"
            "Run notebook 01 (EXP-0) first: it creates the fixed 80/20 "
            "project-grouped holdout split and the dev-only 5-fold manifest "
            "that every later experiment must reuse."
        )

normalized_df = pd.read_parquet(NORMALIZED_DATA_PATH)
manifest_df = pd.read_parquet(MANIFEST_PATH)
holdout_manifest = pd.read_parquet(HOLDOUT_MANIFEST_PATH)

holdout_ids = set(
    holdout_manifest.loc[holdout_manifest["is_holdout"], "source_row_id"]
)
dev_df = normalized_df.loc[
    ~normalized_df["source_row_id"].isin(holdout_ids)
].reset_index(drop=True)
holdout_df = normalized_df.loc[
    normalized_df["source_row_id"].isin(holdout_ids)
].reset_index(drop=True)

assert set(dev_df["project"]).isdisjoint(set(holdout_df["project"]))

print("Normalized dataset:", normalized_df.shape)
print("Dev dataset (80%, used for 5-fold CV):", dev_df.shape)
print("Holdout dataset (20%, untouched until final evaluation):", holdout_df.shape)
print("Manifest:", manifest_df.shape)
print("\nDev projects:", dev_df["project"].nunique())
print("Manifest folds:", sorted(manifest_df["fold"].unique()))
print("Dev vulnerable rate:", f"{dev_df['label'].mean():.4%}")
print("Holdout vulnerable rate:", f"{holdout_df['label'].mean():.4%}")

Normalized dataset: (261667, 5)
Manifest: (261667, 4)
Unique projects: 797
Folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Vulnerable rate: 5.3266%


## Validate dataset against the EXP-0 split

In [ ]:
assert normalized_df["source_row_id"].nunique() == len(normalized_df)
assert normalized_df["code"].notna().all()
assert normalized_df["normalized_code"].notna().all()

assert len(manifest_df) == len(dev_df), (
    "The 5-fold manifest must cover exactly the dev partition."
)
assert manifest_df["source_row_id"].nunique() == len(dev_df)
assert set(manifest_df["fold"].unique()) == {0, 1, 2, 3, 4}

manifest_projects = manifest_df.groupby("fold")["project"].apply(set)
for fold_id in range(5):
    test_projects = manifest_projects.loc[fold_id]
    train_projects = set(
        manifest_df.loc[manifest_df["fold"] != fold_id, "project"]
    )
    assert not test_projects.intersection(train_projects), (
        f"Project overlap detected in fold {fold_id}"
    )

assert set(dev_df["project"]).isdisjoint(set(holdout_df["project"])), (
    "Holdout projects must never appear in the dev partition."
)

print(
    f"✅ EXP-1 uses the same {len(dev_df):,} dev-partition functions and "
    "fixed five-fold manifest as EXP-0."
)
print(
    f"✅ The untouched holdout partition has {len(holdout_df):,} functions "
    "with zero project overlap with dev."
)
print("✅ The fixed five-fold project-aware manifest has zero project overlap.")

✅ EXP-1 uses the same 261,667 functions as EXP-0.
✅ The fixed five-fold project-aware manifest has zero project overlap.


## Smoke-test the static feature extractor

In [ ]:
sample_code = """
int copy_payload(char *dst, const char *src, size_t n) {
    char buffer[64];

    if (n < sizeof(buffer)) {
        memcpy(buffer, src, n);
        buffer[n] = '\\0';
    }

    return 0;
}
"""

sample_features = static_features.extract_static_features(sample_code)

for feature_name in [
    "raw_line_count",
    "identifier_count",
    "if_count",
    "return_count",
    "array_access_count",
    "pointer_declaration_count",
    "memory_api_call_count",
    "sizeof_count",
    "cyclomatic_complexity_proxy",
]:
    print(f"{feature_name:35s}: {sample_features[feature_name]}")

raw_line_count                     : 12.0
identifier_count                   : 22.0
if_count                           : 1.0
return_count                       : 1.0
array_access_count                 : 2.0
pointer_declaration_count          : 2.0
memory_api_call_count              : 1.0
sizeof_count                       : 1.0
cyclomatic_complexity_proxy        : 2.0


## Load or create the static-feature cache

In [ ]:
STATIC_FEATURE_PATH = (
    STATIC_FEATURE_DIR / "cs1_static_features_v1.parquet"
)

STATIC_FEATURE_SUMMARY_PATH = (
    STATIC_FEATURE_DIR / "cs1_static_features_v1_summary.csv"
)

STATIC_FEATURE_METADATA_PATH = (
    STATIC_FEATURE_DIR / "cs1_static_features_v1_metadata.json"
)

if STATIC_FEATURE_PATH.exists():
    static_df = pd.read_parquet(STATIC_FEATURE_PATH)
    print("✅ Loaded existing static-feature cache.")
else:
    print("Static-feature cache not found. Starting one-time extraction...")

    static_config = static_features.StaticFeatureConfig(
        source_id_column="source_row_id",
        code_column="code",
        progress_every=25_000,
    )

    static_df = static_features.extract_static_feature_frame(
        normalized_df[
            [
                "source_row_id",
                "code",
            ]
        ],
        config=static_config,
    )

    static_artifacts = static_features.save_static_feature_artifacts(
        static_frame=static_df,
        output_dir=STATIC_FEATURE_DIR,
        config=static_config,
        source_dataset_path=NORMALIZED_DATA_PATH,
    )

    print("\n✅ Static features extracted and saved:")
    for artifact_name, artifact_path in static_artifacts.items():
        print(f" - {artifact_name}: {artifact_path}")

print("Static feature table shape:", static_df.shape)
display(static_df.head())

✅ Loaded existing static-feature cache.
Static feature table shape: (261667, 55)


,source_row_id,raw_char_count,raw_line_count,nonempty_line_count,avg_nonempty_line_length,max_line_length,comment_char_count,comment_line_count,comment_char_ratio,string_literal_count,...,memory_api_call_count,string_api_call_count,format_api_call_count,input_api_call_count,allocation_api_call_count,deallocation_api_call_count,unsafe_api_presence_count,sizeof_count,null_token_count,assert_call_count
0,0,6098.0,105.0,93.0,64.451613,151.0,52.0,1.0,0.008527,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,1732.0,65.0,55.0,30.327273,71.0,185.0,2.0,0.106813,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,115.0,4.0,4.0,28.000000,59.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,289.0,12.0,10.0,27.800000,50.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,824.0,27.0,23.0,34.695652,78.0,0.0,0.0,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Validate the static-feature cache

In [ ]:
import numpy as np

feature_columns = static_features.FEATURE_COLUMNS

assert len(static_df) == len(normalized_df)
assert static_df["source_row_id"].nunique() == len(normalized_df)
assert list(static_df.columns) == ["source_row_id"] + feature_columns

assert not static_df[feature_columns].isna().any().any()
assert np.isfinite(
    static_df[feature_columns].to_numpy(dtype=float)
).all()
assert (static_df[feature_columns] >= 0).all().all(), (
    "Static count/proxy features must be non-negative."
)

print("✅ Static feature extraction passed all integrity checks.")
print("✅ Rows:", len(static_df))
print("✅ Static features:", len(feature_columns))

✅ Static feature extraction passed all integrity checks.
✅ Rows: 261667
✅ Static features: 54


## Inspect static-feature distributions

In [ ]:
static_summary = static_features.summarize_static_features(static_df)

important_feature_names = [
    "raw_char_count",
    "raw_line_count",
    "cyclomatic_complexity_proxy",
    "function_call_count",
    "pointer_declaration_count",
    "array_access_count",
    "memory_api_call_count",
    "string_api_call_count",
    "format_api_call_count",
    "input_api_call_count",
    "sizeof_count",
]

summary_view = static_summary.loc[
    static_summary["feature"].isin(important_feature_names),
    [
        "feature",
        "mean",
        "50%",
        "95%",
        "max",
        "nonzero_rate",
    ],
].copy()

display(
    summary_view.style.format(
        {
            "mean": "{:.3f}",
            "50%": "{:.3f}",
            "95%": "{:.3f}",
            "max": "{:.3f}",
            "nonzero_rate": "{:.2%}",
        }
    )
)

,feature,mean,50%,95%,max,nonzero_rate
0,raw_char_count,1096.858,475.000,3790.000,240968.000,100.00%
1,raw_line_count,36.566,17.000,123.000,6819.000,100.00%
15,function_call_count,10.398,5.000,35.000,1592.000,99.91%
29,cyclomatic_complexity_proxy,7.023,3.000,24.000,1218.000,100.00%
33,array_access_count,1.561,0.000,7.000,1752.000,26.95%
38,pointer_declaration_count,0.728,0.000,3.000,404.000,30.44%
44,memory_api_call_count,0.211,0.000,1.000,103.000,9.71%
45,string_api_call_count,0.161,0.000,1.000,141.000,5.74%
46,format_api_call_count,0.134,0.000,0.000,251.000,3.73%
47,input_api_call_count,0.013,0.000,0.000,37.000,0.75%


## Configure the EXP-1 Random Forest experiment

In [ ]:
import case_study_1.exp1.exp1_rf as exp1_rf
import case_study_1.evaluation as evaluation

exp1_config = exp1_rf.Exp1Config(
    experiment_name="cs1_exp1_rf_svd_static_oob",

    n_splits=5,
    random_state=42,
    decision_threshold=0.50,

    word_ngram_range=(1, 3),
    word_min_df=3,
    word_max_df=0.995,
    word_max_features=50_000,

    char_analyzer="char",
    char_ngram_range=(3, 4),
    char_min_df=8,
    char_max_df=0.995,
    char_max_features=60_000,

    svd_n_components=256,
    svd_algorithm="randomized",
    svd_n_iter=5,
    svd_n_oversamples=10,

    rf_n_estimators=200,
    rf_criterion="gini",
    rf_max_depth=None,
    rf_min_samples_split=2,
    rf_min_samples_leaf=2,
    rf_max_features="sqrt",
    rf_bootstrap=True,
    rf_max_samples=0.70,
    rf_class_weight="balanced_subsample",
    rf_n_jobs=-1,

    rf_oob_score=True,
    oob_threshold_min=0.005,
    oob_threshold_max=0.250,
    oob_threshold_step=0.005,
    oob_threshold_objective="f1",

    feature_importance_top_n=50,
    verbose=True,
)

print(exp1_config)

Exp1Config(experiment_name='cs1_exp1_rf_svd_static_oob', code_column='normalized_code', source_id_column='source_row_id', label_column='label', project_column='project', fold_column='fold', n_splits=5, random_state=42, decision_threshold=0.5, word_ngram_range=(1, 3), word_min_df=3, word_max_df=0.995, word_max_features=50000, char_analyzer='char', char_ngram_range=(3, 4), char_min_df=8, char_max_df=0.995, char_max_features=60000, lowercase=False, sublinear_tf=True, tfidf_norm='l2', svd_n_components=256, svd_algorithm='randomized', svd_n_iter=5, svd_n_oversamples=10, rf_n_estimators=200, rf_criterion='gini', rf_max_depth=None, rf_min_samples_split=2, rf_min_samples_leaf=2, rf_max_features='sqrt', rf_bootstrap=True, rf_max_samples=0.7, rf_class_weight='balanced_subsample', rf_n_jobs=-1, rf_oob_score=True, oob_threshold_min=0.005, oob_threshold_max=0.25, oob_threshold_step=0.005, oob_threshold_objective='f1', feature_importance_top_n=50, verbose=True)


## Filter static features to the dev partition

In [ ]:
dev_static_df = static_df.loc[
    static_df["source_row_id"].isin(dev_df["source_row_id"])
].reset_index(drop=True)

assert len(dev_static_df) == len(dev_df)
print("Dev static-feature frame:", dev_static_df.shape)

## Profile fold 0 for computational feasibility

In [ ]:
exp1_profile_v2 = exp1_rf.run_exp1_profile_fold(
    normalized_frame=dev_df,
    static_features_frame=dev_static_df,
    manifest=manifest_df,
    fold_id=0,
    config=exp1_config,
)

print("\nFold 0 training and threshold-selection metadata:")
display(
    exp1_profile_v2["training_metadata"][
        [
            "fold",
            "word_tfidf_seconds",
            "char_tfidf_seconds",
            "svd_seconds",
            "model_fit_seconds",
            "total_fold_seconds",
            "selected_oob_threshold",
            "oob_precision",
            "oob_recall",
            "oob_f1",
            "oob_mcc",
            "svd_explained_variance_ratio_sum",
        ]
    ]
)

print("\nHeld-out Fold 0 metrics at the training-OOB selected threshold:")
print(
    evaluation.format_metric_report(
        exp1_profile_v2["profile_metrics"]
    )
)

print("\nDiagnostic only: held-out Fold 0 metrics at fixed threshold 0.50:")
print(
    evaluation.format_metric_report(
        exp1_profile_v2["default_threshold_metrics"]
    )
)

[21:13:27] CS1-EXP1 profiling mode: running Fold 1/5 only.
[21:13:30] Fold 1/5 started | train=192,411, test=69,256, train projects=743, test projects=54.
[21:13:30] Fold 1/5 | fitting word TF-IDF...
[21:16:46] Fold 1/5 | word TF-IDF done in 3.27 min (50,000 features).
[21:16:46] Fold 1/5 | fitting character TF-IDF...
[21:22:53] Fold 1/5 | character TF-IDF done in 6.12 min (60,000 features).
[21:22:53] Fold 1/5 | joining sparse lexical matrices...
[21:22:55] Fold 1/5 | sparse lexical matrix ready in 2.1s (110,000 columns).
[21:22:56] Fold 1/5 | fitting train-only TruncatedSVD (256 components)...
[21:30:13] Fold 1/5 | TruncatedSVD done in 7.29 min (explained variance ratio sum=0.2077).
[21:30:13] Fold 1/5 | loading cached static features...
[21:30:13] Fold 1/5 | combining SVD and static features...
[21:30:13] Fold 1/5 | training Random Forest (trees=200, max_features=sqrt, class_weight=balanced_subsample)...
[21:47:40] Fold 1/5 | Random Forest done in 17.45 min.
[21:47:40] Fold 1/5 | se

,fold,word_tfidf_seconds,char_tfidf_seconds,svd_seconds,model_fit_seconds,total_fold_seconds,selected_oob_threshold,oob_precision,oob_recall,oob_f1,oob_mcc,svd_explained_variance_ratio_sum
0,0,196.010608,366.973231,437.32291,1046.856106,2061.443788,0.13,0.190688,0.389716,0.256078,0.212063,0.207744



Held-out Fold 0 metrics at the training-OOB selected threshold:
Pooled Out-of-Fold Evaluation
                   n_samples: 69256
                vulnerable_1: 3397
            non_vulnerable_0: 65859
               positive_rate: 0.049050
                   threshold: 0.130000
    average_precision_pr_auc: 0.107073
                   precision: 0.148754
                      recall: 0.168678
                          f1: 0.158091
                         mcc: 0.112035
                 specificity: 0.950212
         false_positive_rate: 0.049788
               true_negative: 62580
              false_positive: 3279
              false_negative: 2824
               true_positive: 573

Diagnostic only: held-out Fold 0 metrics at fixed threshold 0.50:
Pooled Out-of-Fold Evaluation
                   n_samples: 69256
                vulnerable_1: 3397
            non_vulnerable_0: 65859
               positive_rate: 0.049050
                   threshold: 0.500000
    average_precision_pr_

## Run the official EXP-1 five-fold experiment

In [ ]:
from pathlib import Path
import pandas as pd

EXP1_OFFICIAL_OUTPUT_DIR = (
    OUTPUT_DIR / "official_svd_static_rf_oob_v2"
)

EXP1_OFFICIAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

existing_files = list(EXP1_OFFICIAL_OUTPUT_DIR.iterdir())

if existing_files:
    raise RuntimeError(
        "Official EXP-1 output folder is not empty. "
        "Use a fresh directory or deliberately remove only incomplete artifacts."
    )

exp1_results = exp1_rf.run_exp1(
    normalized_frame=dev_df,
    static_features_frame=dev_static_df,
    manifest=manifest_df,
    config=exp1_config,
    output_dir=EXP1_OFFICIAL_OUTPUT_DIR,
    additional_metadata={
        "run_type": "official_full_5fold_oof_evaluation_on_dev_partition",
        "normalized_dataset_path": str(NORMALIZED_DATA_PATH),
        "static_feature_cache_path": str(STATIC_FEATURE_PATH),
        "manifest_path": str(MANIFEST_PATH),
        "split_protocol": (
            "Fixed 80/20 project-grouped holdout, then 5-fold StratifiedGroupKFold "
            "manifest on the 80% dev partition, grouped by project, random_state=42 "
            "(identical split reused from EXP-0)."
        ),
        "representation": (
            "Word TF-IDF (1,3) + Character TF-IDF (3,4), "
            "train-only TruncatedSVD(256), plus 54 deterministic "
            "source-level static proxy features."
        ),
        "classifier": (
            "RandomForestClassifier with 200 trees, balanced_subsample "
            "class weights, and 70% bootstrap samples."
        ),
        "threshold_protocol": (
            "Per-fold threshold selected using training-only Random Forest "
            "out-of-bag predictions, maximizing F1 on the predefined "
            "0.005–0.250 threshold grid."
        ),
        "pilot_note": (
            "Fold 0 computational profiles were used to validate feasibility "
            "and the threshold-selection protocol before the official full run."
        ),
    },
)

[22:13:29] CS1-EXP1 official run started: 5-fold grouped CV.
[22:13:29] Configuration: lexical<= 110,000, SVD=256, static=54, RF trees=200.
[22:13:36] Fold 1/5 started | train=192,411, test=69,256, train projects=743, test projects=54.
[22:13:36] Fold 1/5 | fitting word TF-IDF...
[22:16:48] Fold 1/5 | word TF-IDF done in 3.21 min (50,000 features).
[22:16:48] Fold 1/5 | fitting character TF-IDF...
[22:22:55] Fold 1/5 | character TF-IDF done in 6.11 min (60,000 features).
[22:22:55] Fold 1/5 | joining sparse lexical matrices...
[22:22:57] Fold 1/5 | sparse lexical matrix ready in 2.0s (110,000 columns).
[22:22:57] Fold 1/5 | fitting train-only TruncatedSVD (256 components)...
[22:30:02] Fold 1/5 | TruncatedSVD done in 7.09 min (explained variance ratio sum=0.2077).
[22:30:02] Fold 1/5 | loading cached static features...
[22:30:02] Fold 1/5 | combining SVD and static features...
[22:30:03] Fold 1/5 | training Random Forest (trees=200, max_features=sqrt, class_weight=balanced_subsample)..

## Refit the final model and evaluate on the holdout

In [ ]:
EXP1_HOLDOUT_OUTPUT_DIR = (
    OUTPUT_DIR / "final_holdout_svd_static_rf_oob_v2"
)
EXP1_HOLDOUT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

exp1_holdout_results = exp1_rf.run_exp1_final_holdout(
    dev_frame=dev_df,
    static_features_frame=static_df,
    holdout_frame=holdout_df,
    config=exp1_config,
    output_dir=EXP1_HOLDOUT_OUTPUT_DIR,
    additional_metadata={
        "run_type": "final_model_selected_holdout_evaluation",
        "normalized_dataset_path": str(NORMALIZED_DATA_PATH),
        "static_feature_cache_path": str(STATIC_FEATURE_PATH),
        "holdout_manifest_path": str(HOLDOUT_MANIFEST_PATH),
        "split_protocol": (
            "Fixed 80/20 project-grouped holdout (identical split reused from "
            "EXP-0); final model fit on the full dev partition, scored on the "
            "untouched holdout exactly once."
        ),
    },
)

print("\nFinal EXP-1 holdout metrics:")
print(
    evaluation.format_metric_report(
        exp1_holdout_results["metrics"]
    )
)
print(
    f"\nSelected operating threshold (from full-dev OOB): "
    f"{exp1_holdout_results['metrics'].get('threshold', 'n/a')}"
)

## List saved holdout run artifacts

In [ ]:
from pathlib import Path
import pandas as pd
import json

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData"
)

RUN_DIR = (
    DRIVE_ROOT
    / "outputs"
    / "cs1_exp1_rf"
    / "official_svd_static_rf_oob_v2"
)

assert RUN_DIR.exists(), f"Run folder not found: {RUN_DIR}"

saved_files = sorted(
    [path for path in RUN_DIR.iterdir() if path.is_file()],
    key=lambda path: path.name.lower(),
)

print("Run directory:")
print(RUN_DIR)

print(f"\nSaved files found: {len(saved_files)}")

file_inventory = pd.DataFrame(
    [
        {
            "filename": path.name,
            "extension": path.suffix,
            "size_kb": round(path.stat().st_size / 1024, 2),
        }
        for path in saved_files
    ]
)

display(file_inventory)

Run directory:
/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_exp1_rf/official_svd_static_rf_oob_v2

Saved files found: 17


,filename,extension,size_kb
0,cs1_exp1_rf_svd_static_oob_config.json,.json,1.33
1,cs1_exp1_rf_svd_static_oob_evaluation_metadata...,.json,3.89
2,cs1_exp1_rf_svd_static_oob_feature_importances...,.csv,90.36
3,cs1_exp1_rf_svd_static_oob_fold_metrics.csv,.csv,1.81
4,cs1_exp1_rf_svd_static_oob_fold_summary.csv,.csv,1.25
5,cs1_exp1_rf_svd_static_oob_fold_training.csv,.csv,3.05
6,cs1_exp1_rf_svd_static_oob_oob_operating_fold_...,.csv,1.63
7,cs1_exp1_rf_svd_static_oob_oob_operating_fold_...,.csv,0.71
8,cs1_exp1_rf_svd_static_oob_oob_operating_poole...,.json,0.83
9,cs1_exp1_rf_svd_static_oob_oof_predictions.csv,.csv,11852.13


## Load and display pooled holdout metrics

In [ ]:
metric_json_paths = sorted(
    RUN_DIR.glob("*pooled_metrics*.json"),
    key=lambda path: path.name.lower(),
)

if not metric_json_paths:
    raise FileNotFoundError(
        "No pooled-metrics JSON file was found in the output folder."
    )

loaded_pooled_metrics = {}

for metric_path in metric_json_paths:
    with open(metric_path, "r", encoding="utf-8") as file:
        metrics = json.load(file)

    loaded_pooled_metrics[metric_path.name] = metrics

    print("\n" + "=" * 90)
    print(metric_path.name)
    print("=" * 90)

    important_keys = [
        "n_samples",
        "vulnerable_1",
        "non_vulnerable_0",
        "positive_rate",
        "threshold",
        "average_precision_pr_auc",
        "precision",
        "recall",
        "f1",
        "mcc",
        "specificity",
        "false_positive_rate",
        "true_negative",
        "false_positive",
        "false_negative",
        "true_positive",
        "predicted_positive",
        "predicted_positive_rate",
        "selected_threshold_min",
        "selected_threshold_max",
        "selected_threshold_mean",
    ]

    for key in important_keys:
        if key in metrics:
            print(f"{key:>32}: {metrics[key]}")


cs1_exp1_rf_svd_static_oob_oob_operating_pooled_metrics.json
                       n_samples: 261667
                    vulnerable_1: 13938
                non_vulnerable_0: 247729
                   positive_rate: 0.0532661741832176
                       threshold: None
        average_precision_pr_auc: 0.1367543133327688
                       precision: 0.16197297078578943
                          recall: 0.2872004591763524
                              f1: 0.2071302907999586
                             mcc: 0.15633543047723225
                     specificity: 0.9163965462259163
             false_positive_rate: 0.08360345377408378
                   true_negative: 227018
                  false_positive: 20711
                  false_negative: 9935
                   true_positive: 4003
              predicted_positive: 24714
         predicted_positive_rate: 0.09444828732702251
          selected_threshold_min: 0.125
          selected_threshold_max: 0.135
         selected

## Load and display fold metrics

In [ ]:
fold_metric_paths = sorted(
    RUN_DIR.glob("*fold_metrics*.csv"),
    key=lambda path: path.name.lower(),
)

if not fold_metric_paths:
    print("⚠️ No fold_metrics CSV files found.")
else:
    for fold_path in fold_metric_paths:
        fold_df = pd.read_csv(fold_path)

        print("\n" + "=" * 90)
        print(fold_path.name)
        print("=" * 90)
        print("Shape:", fold_df.shape)
        print("Columns:", fold_df.columns.tolist())

        preferred_columns = [
            "fold",
            "threshold",
            "n_samples",
            "test_unique_projects",
            "positive_rate",
            "average_precision_pr_auc",
            "precision",
            "recall",
            "f1",
            "mcc",
            "false_positive_rate",
            "predicted_positive",
            "predicted_positive_rate",
        ]

        available_columns = [
            column for column in preferred_columns
            if column in fold_df.columns
        ]

        display(fold_df[available_columns])


cs1_exp1_rf_svd_static_oob_fold_metrics.csv
Shape: (5, 24)
Columns: ['fold', 'n_samples', 'vulnerable_1', 'non_vulnerable_0', 'positive_rate', 'threshold', 'average_precision_pr_auc', 'precision', 'recall', 'f1', 'mcc', 'accuracy', 'balanced_accuracy', 'specificity', 'negative_predictive_value', 'false_positive_rate', 'false_negative_rate', 'true_negative', 'false_positive', 'false_negative', 'true_positive', 'predicted_positive', 'predicted_positive_rate', 'test_unique_projects']


,fold,threshold,n_samples,test_unique_projects,positive_rate,average_precision_pr_auc,precision,recall,f1,mcc,false_positive_rate,predicted_positive,predicted_positive_rate
0,0,0.5,69256,54,0.049050,0.107073,0.750000,0.000883,0.001764,0.024666,0.000015,4,0.000058
1,1,0.5,51771,188,0.054046,0.141368,0.439024,0.019299,0.036974,0.083090,0.001409,123,0.002376
2,2,0.5,46043,192,0.050496,0.123734,0.285714,0.003441,0.006800,0.026499,0.000457,28,0.000608
3,3,0.5,47064,181,0.056668,0.164880,0.333333,0.001500,0.002986,0.019110,0.000180,12,0.000255
4,4,0.5,47533,182,0.057876,0.158212,0.500000,0.018539,0.035752,0.087803,0.001139,102,0.002146



cs1_exp1_rf_svd_static_oob_oob_operating_fold_metrics.csv
Shape: (5, 22)
Columns: ['n_samples', 'vulnerable_1', 'non_vulnerable_0', 'positive_rate', 'threshold', 'average_precision_pr_auc', 'precision', 'recall', 'f1', 'mcc', 'specificity', 'negative_predictive_value', 'false_positive_rate', 'false_negative_rate', 'true_negative', 'false_positive', 'false_negative', 'true_positive', 'predicted_positive', 'predicted_positive_rate', 'fold', 'test_unique_projects']


,fold,threshold,n_samples,test_unique_projects,positive_rate,average_precision_pr_auc,precision,recall,f1,mcc,false_positive_rate,predicted_positive,predicted_positive_rate
0,0,0.130,69256,54,0.049050,0.107073,0.148754,0.168678,0.158091,0.112035,0.049788,3852,0.055620
1,1,0.125,51771,188,0.054046,0.141368,0.147706,0.371694,0.211404,0.164346,0.122537,7041,0.136003
2,2,0.135,46043,192,0.050496,0.123734,0.144588,0.297634,0.194628,0.146357,0.093646,4786,0.103946
3,3,0.125,47064,181,0.056668,0.164880,0.183298,0.384327,0.248214,0.201115,0.102867,5592,0.118817
4,4,0.135,47533,182,0.057876,0.158212,0.195469,0.244638,0.217307,0.164662,0.061855,3443,0.072434


## Load and display fold training details

In [ ]:
training_paths = sorted(
    RUN_DIR.glob("*fold_training*.csv"),
    key=lambda path: path.name.lower(),
)

if not training_paths:
    print("⚠️ No fold_training CSV file found.")
else:
    for training_path in training_paths:
        training_df = pd.read_csv(training_path)

        print("\n" + "=" * 90)
        print(training_path.name)
        print("=" * 90)

        preferred_columns = [
            "fold",
            "train_rows",
            "test_rows",
            "word_tfidf_seconds",
            "char_tfidf_seconds",
            "svd_seconds",
            "model_fit_seconds",
            "total_fold_seconds",
            "selected_oob_threshold",
            "oob_precision",
            "oob_recall",
            "oob_f1",
            "oob_mcc",
            "svd_explained_variance_ratio_sum",
        ]

        available_columns = [
            column for column in preferred_columns
            if column in training_df.columns
        ]

        display(training_df[available_columns])

        if "total_fold_seconds" in training_df.columns:
            total_minutes = training_df["total_fold_seconds"].sum() / 60
            print(f"\nTotal five-fold runtime: {total_minutes:.2f} minutes")


cs1_exp1_rf_svd_static_oob_fold_training.csv


,fold,train_rows,test_rows,word_tfidf_seconds,char_tfidf_seconds,svd_seconds,model_fit_seconds,total_fold_seconds,selected_oob_threshold,oob_precision,oob_recall,oob_f1,oob_mcc,svd_explained_variance_ratio_sum
0,0,192411,69256,192.413418,366.879402,425.282092,1040.152590,2038.812225,0.130,0.190688,0.389716,0.256078,0.212063,0.207744
1,1,209896,51771,186.333536,365.252327,452.330852,1191.911923,2210.488240,0.125,0.190126,0.401346,0.258022,0.217154,0.209881
2,2,215624,46043,199.331893,371.257336,472.329987,1190.770977,2248.511237,0.135,0.192221,0.375786,0.254342,0.210267,0.209126
3,3,214603,47064,195.448824,374.663662,470.934368,1182.305857,2237.616939,0.125,0.182512,0.381865,0.246980,0.204747,0.209921
4,4,214134,47533,194.318317,371.673009,467.517335,1216.537086,2264.690435,0.135,0.192564,0.372665,0.253921,0.211490,0.205775



Total five-fold runtime: 183.34 minutes


## Load and display feature importances

In [ ]:
importance_paths = sorted(
    RUN_DIR.glob("*feature_importances*.csv"),
    key=lambda path: path.name.lower(),
)

if not importance_paths:
    print("⚠️ No feature importance CSV file found.")
else:
    full_importance_paths = [
        path for path in importance_paths
        if "static_feature_importances" not in path.name.lower()
    ]

    if not full_importance_paths:
        full_importance_paths = importance_paths

    importance_path = full_importance_paths[0]
    importance_df = pd.read_csv(importance_path)

    print("Using:", importance_path.name)
    print("Shape:", importance_df.shape)

    importance_by_group = (
        importance_df
        .groupby(["fold", "feature_group"], as_index=False)
        .agg(
            total_importance=("importance", "sum"),
        )
    )

    importance_by_group["importance_share"] = (
        importance_by_group
        .groupby("fold")["total_importance"]
        .transform(lambda values: values / values.sum())
    )

    print("\nImportance share by fold:")
    display(importance_by_group)

    average_importance_by_group = (
        importance_by_group
        .groupby("feature_group", as_index=False)
        .agg(
            mean_importance_share=("importance_share", "mean"),
            std_importance_share=("importance_share", "std"),
        )
    )

    print("\nAverage importance across all folds:")
    display(
        average_importance_by_group.style.format(
            {
                "mean_importance_share": "{:.2%}",
                "std_importance_share": "{:.2%}",
            }
        )
    )

Using: cs1_exp1_rf_svd_static_oob_feature_importances.csv
Shape: (1550, 5)

Importance share by fold:


,fold,feature_group,total_importance,importance_share
0,0,static,0.217796,0.217796
1,0,svd_component,0.782204,0.782204
2,1,static,0.208566,0.208566
3,1,svd_component,0.791434,0.791434
4,2,static,0.209323,0.209323
5,2,svd_component,0.790677,0.790677
6,3,static,0.198097,0.198097
7,3,svd_component,0.801903,0.801903
8,4,static,0.215899,0.215899
9,4,svd_component,0.784101,0.784101



Average importance across all folds:


,feature_group,mean_importance_share,std_importance_share
0,static,20.99%,0.77%
1,svd_component,79.01%,0.77%
